# ColumnTransformer

In this notebook, we will learn:
- Why we need `ColumnTransformer`
- How preprocessing was done before it
- The practical use of `ColumnTransformer`
- A clear, step-by-step code example with model training

## 1) Why do we need ColumnTransformer?

Real-world datasets usually have mixed column types:
- Numerical columns (age, salary, marks)
- Categorical columns (city, gender, product type)

Different preprocessing is needed for different columns:
- Numerical -> scaling (like `StandardScaler`)
- Categorical -> encoding (like `OneHotEncoder`)

`ColumnTransformer` lets us apply the right transformer to the right columns in one unified step. This prevents confusion and keeps preprocessing consistent for train/test/new data.

## 2) How was it done before ColumnTransformer?

Earlier, we often did preprocessing manually:
1. Select categorical columns and apply `pd.get_dummies()`
2. Manually scale numerical columns
3. Make sure train and test have same columns
4. Handle unseen categories manually

This approach works for small demos, but for real projects it is error-prone and hard to maintain.

In [1]:
import pandas as pd

# Small mixed-type dataset (numerical + categorical)
df = pd.DataFrame({
    "age": [25, 32, 47, 51, 62, 23, 40, 36],
    "salary": [30000, 50000, 70000, 75000, 90000, 28000, 62000, 52000],
    "city": ["Delhi", "Mumbai", "Delhi", "Pune", "Mumbai", "Pune", "Delhi", "Mumbai"],
    "gender": ["M", "F", "F", "M", "M", "F", "M", "F"],
    "bought": [0, 1, 1, 1, 1, 0, 1, 0]
})

df.head()

,age,salary,city,gender,bought
0,25,30000,Delhi,M,0
1,32,50000,Mumbai,F,1
2,47,70000,Delhi,F,1
3,51,75000,Pune,M,1
4,62,90000,Mumbai,M,1


In [2]:
# Manual old approach (for understanding only)
X_manual = df.drop("bought", axis=1)
y = df["bought"]

# 1) Encode categorical columns manually
X_manual_encoded = pd.get_dummies(X_manual, columns=["city", "gender"], drop_first=True)

# 2) Manual scaling (example)
for col in ["age", "salary"]:
    mean = X_manual_encoded[col].mean()
    std = X_manual_encoded[col].std()
    X_manual_encoded[col] = (X_manual_encoded[col] - mean) / std

print("Manual preprocessed shape:", X_manual_encoded.shape)
X_manual_encoded.head()

Manual preprocessed shape: (8, 5)


,age,salary,city_Mumbai,city_Pune,gender_M
0,-1.086821,-1.258690,False,False,True
1,-0.562149,-0.330624,True,False,False
2,0.562149,0.597443,False,False,False
3,0.861961,0.829459,False,True,True
4,1.686446,1.525509,True,False,True


## 3) ColumnTransformer approach (recommended)

Now we apply:
- `StandardScaler` on numerical columns
- `OneHotEncoder` on categorical columns

And combine preprocessing + model in one `Pipeline`.

This is cleaner, safer, and production-friendly.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df.drop("bought", axis=1)
y = df["bought"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# transformer
transformer = ColumnTransformer(
    transformers=[
        ("tf1", OneHotEncoder(drop="first", handle_unknown="ignore"), ["city", "gender"]),
        ("tf2", StandardScaler(), ["age", "salary"]),
    ],
    remainder="passthrough"
)

# 1) Fit transformer on train data only
# 2) Transform train and test data
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

print("Original train shape:", X_train.shape)
print("Transformed train shape:", X_train_transformed.shape)

# Train model on transformed data (still no pipeline)
model = LogisticRegression()
model.fit(X_train_transformed, y_train)

# Predict and evaluate
y_pred = model.predict(X_test_transformed)
print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

Original train shape: (6, 4)
Transformed train shape: (6, 5)

Accuracy: 0.5

Classification Report:

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



In [4]:
# See transformed feature names directly from your fitted transformer
feature_names = transformer.get_feature_names_out()

print("Total transformed features:", len(feature_names))
print(feature_names)

# Optional: make transformed train data readable as a DataFrame
X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_train_transformed_df.head()

Total transformed features: 5
['tf1__city_Mumbai' 'tf1__city_Pune' 'tf1__gender_M' 'tf2__age'
 'tf2__salary']


,tf1__city_Mumbai,tf1__city_Pune,tf1__gender_M,tf2__age,tf2__salary
0,0.0,0.0,1.0,-1.581043,-1.759134
1,1.0,0.0,0.0,-0.640963,-0.592271
2,0.0,0.0,0.0,0.299116,0.362435
3,1.0,0.0,1.0,1.581043,1.423219
4,0.0,1.0,1.0,0.640963,0.627631


## 4) Final use of ColumnTransformer

`ColumnTransformer` is useful because it:
- Applies different preprocessing to different column types in one place
- Avoids manual mistakes in train/test preprocessing
- Works smoothly with `Pipeline` and model training
- Is easier to deploy in real ML workflows

In short: it makes preprocessing organized, repeatable, and reliable.